# 🎓 1st Project — **2nd Delivery**


### Teacher
- Cristian Camilo Zapata Zuluaga

### Students
- Roi Jared Flores Garza Stone
- Ivan Morales

November 5th, 2025, ITESO

# Model Training

Working with Mlflow through DataBricks

In [1]:
import os, mlflow
from dotenv import load_dotenv

load_dotenv(override=True) # Cargar las variables de entorno desde el archivo .env
EXPERIMENT_NAME = "/Users/roiflores.2213@gmail.com/coffee-intake-experiments" 

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

## Import required packages

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score


import pandas as pd
import numpy as np
from functools import partial
import optuna
from optuna.integration import MLflowCallback

import joblib
import importlib
import sys
import os

c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Preprocessor importing

We will import the preprocessing pipeline done in ```_02_dat_wrangling``` to process the testing data the ml models will use

In [3]:
path_a_la_carpeta_scripts = os.path.abspath(os.path.join(os.getcwd(), '..', 'scripts')) #Finds the scripts directory
if path_a_la_carpeta_scripts not in sys.path:
    sys.path.append(path_a_la_carpeta_scripts) 
    
preprocessing = importlib.import_module("preprocessing") # Imports the module
optuna_utils = importlib.import_module("optuna_utils")

full_pipeline = joblib.load("../models/preprocessing_pipeline.joblib") #Imports the pipeline object

c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Data importing

In [4]:
df = pd.read_csv("../data/raw/synthetic_coffee_health_10000.csv")

X_train, X_test, y_train, y_test = train_test_split(df.drop("Sleep_Quality", axis=1), df["Sleep_Quality"],
                                                    test_size=0.2, random_state=88, stratify=df[["Sleep_Quality"]])

## Hyperparameter tuning

Here we create a configuration in which we signall which object will be used for each type of model

In [5]:
model_config = {
    "LogisticRegression": {"model_class":LogisticRegression(random_state=42)},
    "RandomForest": {"model_class": RandomForestClassifier(random_state=42)},
    "MLP": {"model_class": MLPClassifier(random_state=42)}
}
MAX_EVALS_PER_MODEL=15

In [6]:
from sklearn.model_selection import cross_val_score

columns_to_drop = [
    "Coffee_Intake",
    "Gender_Other",
    "Occupation_Other",
    "is_oceania",
    "Health_Issues",
    "Stress_Level",
]

def objective(trial, model_class, model_name):
    
    full_params = optuna_utils.define_search_space(trial, model_name) #Uses the parameter space from optuna_utils.py
    
    model = model_class.set_params(**full_params) # Unpacks the parameters and sets up the model with those
    
    preprocessing_pipeline = preprocessing.create_preprocessing_pipeline(columns_to_drop) # Creates a pipeline using the preprocessing seen before
    
    full_trial_pipeline = Pipeline([
        ("preprocessor", preprocessing_pipeline), 
        ("model", model)
    ])
    
    score = cross_val_score(full_trial_pipeline, X_train, y_train,       # Cross validates the model with the parameter and returns the score
                            cv=3, scoring="f1_weighted", error_score="raise").mean() 
    
    return score

In [8]:
from mlflow.models import infer_signature

notebook_dir = os.getcwd() # Gets the notebook's direction
SCRIPT_DIR = os.path.abspath(os.path.join(notebook_dir, "..", "scripts")) # Getts the direction of the dir scripts

for model_name, config in model_config.items():
    with mlflow.start_run(run_name=f"{model_name}_HPO") as parent_run:
        
        kwargs = {"nested": True} #Indica que el run es nested
        
        mlflow_callback = MLflowCallback(
            tracking_uri="databricks", #Llama a databricks
            metric_name="f1_score", #Uses f1_score as metric
            create_experiment=False, # Doesn't create a new experiment
            mlflow_kwargs= kwargs)
        
        study = optuna.create_study(direction="maximize") # We want to maximize the f1_score
        
        obj_func = partial(
            objective, # Goes into objective function
            model_class=config["model_class"],
            model_name=model_name
        )
        
        study.optimize(
            obj_func,
            n_trials=MAX_EVALS_PER_MODEL,
            callbacks=[mlflow_callback]
        )
        
        best_f1_metric = study.best_value #Gets best f1_score
        best_params_cleaned = study.best_params #Gets best params
        
        print(f"--- 🏆 Mejor F1 para {model_name}: {best_f1_metric:.4f} ---")
        
        # --------------------------------------------------------------
        # This block of code sets up the fixed parameters for each model
        if model_name == 'LogisticRegression':
            best_params_cleaned['solver'] = 'saga'
            best_params_cleaned['max_iter'] = 1000
            
        elif model_name == 'MLP':
            best_params_cleaned['solver'] = 'adam'
            best_params_cleaned['max_iter'] = 500
        # --------------------------------------------------------------
        
        best_model_instance = config["model_class"].set_params(**best_params_cleaned) # Chooses the best model and its parameters
        
        # Creates a new pipeline to preprocess and fit with the whole data 
        final_preprocessing = preprocessing.create_preprocessing_pipeline(columns_to_drop) 
        
        final_production_pipeline = Pipeline([
            ("preprocessor", final_preprocessing),
            ("model", best_model_instance)
        ])
        final_production_pipeline.fit(X_train, y_train) # Re-trains the model, but now with all of the data
        
        example_data = X_train.iloc[:10] #Input data example
        example_predictions = final_production_pipeline.predict(example_data) # Output data example
        signature = infer_signature(example_data, example_predictions) #Creates signature for the model
        
        # 8. Loguea el pipeline final y la métrica en el Run Padre
        mlflow.log_params(best_params_cleaned) # Loguea los params limpios en el padre
        mlflow.log_metric("F1_score", best_f1_metric)
        
        code_path_preprocessing = os.path.join(SCRIPT_DIR, "preprocessing.py") #Gets the preprocessing.py path
        code_path_optuna = os.path.join(SCRIPT_DIR, "optuna_utils.py") #Gets the optuna_utils.py path
                
        mlflow.sklearn.log_model(
            sk_model=final_production_pipeline,
            artifact_path="model",
            code_paths=[code_path_preprocessing, code_path_optuna],
            signature=signature,
            input_example=example_data
        )

C:\Users\Roi_f\AppData\Local\Temp\ipykernel_27660\3115669844.py:11: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
[I 2025-11-17 23:29:41,386] A new study created in memory with name: no-name-e3044277-b48f-41f5-b3dd-cd75ec558e3f
c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
[I 2025-11-17 23:30:02,905] Trial 0 finished with value: 0.971217

🏃 View run 0 at: https://dbc-8e655ea0-6144.cloud.databricks.com/ml/experiments/567513252354655/runs/acd4186723654d66ab10686c0ae4f24d
🧪 View experiment at: https://dbc-8e655ea0-6144.cloud.databricks.com/ml/experiments/567513252354655


[I 2025-11-17 23:30:08,283] Trial 1 finished with value: 0.9710253113316716 and parameters: {'C': 0.17522209472465547, 'penalty': 'l1'}. Best is trial 0 with value: 0.9712178052815997.


🏃 View run 1 at: https://dbc-8e655ea0-6144.cloud.databricks.com/ml/experiments/567513252354655/runs/199bdace8a174916a76f7b9168f98d21
🧪 View experiment at: https://dbc-8e655ea0-6144.cloud.databricks.com/ml/experiments/567513252354655


[I 2025-11-17 23:30:10,184] Trial 2 finished with value: 0.8981729884368675 and parameters: {'C': 0.049298978333090236, 'penalty': 'l2'}. Best is trial 0 with value: 0.9712178052815997.


...

🏃 View run 13 at: https://dbc-8e655ea0-6144.cloud.databricks.com/ml/experiments/567513252354655/runs/fd15c81471af4abda5b998e4adf48c56
🧪 View experiment at: https://dbc-8e655ea0-6144.cloud.databricks.com/ml/experiments/567513252354655


c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (50,) which is of type tuple.
  warnings.warn(message)
c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (100,) which is of type tuple.
  warnings.warn(message)
c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (50, 50) which is of type tuple.
  warnings.warn(message)
c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\optuna\distributions.py:502: UserWarning: Choices for a categori

🏃 View run 14 at: https://dbc-8e655ea0-6144.cloud.databricks.com/ml/experiments/567513252354655/runs/86d7f42af86c4c2c9c69fffc572b5de1
🧪 View experiment at: https://dbc-8e655ea0-6144.cloud.databricks.com/ml/experiments/567513252354655
--- 🏆 Mejor F1 para MLP: 0.9714 ---


c:\Users\Roi_f\PCD\coffee_intake_project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/17 23:38:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/17 23:38:25 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added

🏃 View run MLP_HPO at: https://dbc-8e655ea0-6144.cloud.databricks.com/ml/experiments/567513252354655/runs/bcb333d40e5a4b5ca45583ce4db48bd9
🧪 View experiment at: https://dbc-8e655ea0-6144.cloud.databricks.com/ml/experiments/567513252354655


## Creating the challenger and the champion

In [ ]:
model_registry = "workspace.default.coffee-intake-experiments" 
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by= ["metrics.F1_score DESC"], # Get the best f1_score
    output_format="list"
)

if len(runs) > 0:
    best_run = runs[0] # This has the best f1_score
    second_best = runs[1] # this has the second best f1_score

After sorting for the best and second best model, we will register those models

In [10]:
result_champ = mlflow.register_model(
    model_uri=f"runs:/{best_run.info.run_id}/model",
    name=model_registry
)

result_chall = mlflow.register_model(
    model_uri=f"runs:/{second_best.info.run_id}/model",
    name=model_registry
)

Registered model 'workspace.default.coffee-intake-experiments' already exists. Creating a new version of this model...
2025/11/17 23:39:21 WARNING mlflow.tracking._model_registry.fluent: Run with id ec4cdcab89554ec6a9b23caec418991f has no artifacts at artifact path 'model', registering model based on models:/m-b01def09d1044e2c871e469a3fa57998 instead
Uploading artifacts: 100%|██████████| 10/10 [00:04<00:00,  2.03it/s]
Created version '1' of model 'workspace.default.coffee-intake-experiments'.
Registered model 'workspace.default.coffee-intake-experiments' already exists. Creating a new version of this model...
2025/11/17 23:39:36 WARNING mlflow.tracking._model_registry.fluent: Run with id 8af6b5783656479a8135dd31bff17ca3 has no artifacts at artifact path 'model', registering model based on models:/m-6f13814d2a974cf780cb997b2a1ce145 instead
Uploading artifacts: 100%|██████████| 11/11 [00:04<00:00,  2.44it/s]
Created version '2' of model 'workspace.default.coffee-intake-experiments'.


Finally, we will asign the alias ```Champion``` and ```Challenger```

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()
model_chall_version = result_chall.version # Callenger version
model_champ_version = result_champ.version # Champion version
challenger_alias ="Challenger"
champ_alias ="Champion"

# Challenger alias setter
client.set_registered_model_alias(
    name=model_registry,
    alias=challenger_alias,
    version=model_chall_version
)

# Champion alias
client.set_registered_model_alias(
    name=model_registry,
    alias=champ_alias,
    version= model_champ_version
)